In [ ]:
#python/Spark/text_search.py

In [6]:
from pyspark import SparkConf, SparkContext
import re
import sys

In [7]:
# Create the SparkContext
#    sc = SparkContext(appName='SparkWordCount')

conf = SparkConf().setMaster("local").setAppName("SparkWordCount")
sc = SparkContext(conf = conf)

In [8]:
# Introduce a key word to search
keyword = input("Introduce the key-word : ")

Introduce the key-word :  father


In [10]:
# Broadcast the requested term
#   requested_movie = sc.broadcast(sys.argv[1])
requested_movie = sc.broadcast(keyword)

In [11]:
# Load the input file
source_file = sc.textFile('/ml-25/movies.csv')
source_file.take(4)

['movieId,title,genres',
 '1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy',
 '2,Jumanji (1995),Adventure|Children|Fantasy',
 '3,Grumpier Old Men (1995),Comedy|Romance']

In [12]:
# Get the movie title from the second field
titles = source_file.map(lambda line: line.split(',')[1])

In [13]:
titles.take(10)

['title',
 'Toy Story (1995)',
 'Jumanji (1995)',
 'Grumpier Old Men (1995)',
 'Waiting to Exhale (1995)',
 'Father of the Bride Part II (1995)',
 'Heat (1995)',
 'Sabrina (1995)',
 'Tom and Huck (1995)',
 'Sudden Death (1995)']

In [14]:
# Create a map of the normalized title to the raw title 
normalized_title = titles.map(lambda title: (re.sub(r'\s*\(\d{4}\)','', title).lower(), title))

In [15]:
normalized_title.take(20)

[('title', 'title'),
 ('toy story', 'Toy Story (1995)'),
 ('jumanji', 'Jumanji (1995)'),
 ('grumpier old men', 'Grumpier Old Men (1995)'),
 ('waiting to exhale', 'Waiting to Exhale (1995)'),
 ('father of the bride part ii', 'Father of the Bride Part II (1995)'),
 ('heat', 'Heat (1995)'),
 ('sabrina', 'Sabrina (1995)'),
 ('tom and huck', 'Tom and Huck (1995)'),
 ('sudden death', 'Sudden Death (1995)'),
 ('goldeneye', 'GoldenEye (1995)'),
 ('"american president', '"American President'),
 ('dracula: dead and loving it', 'Dracula: Dead and Loving It (1995)'),
 ('balto', 'Balto (1995)'),
 ('nixon', 'Nixon (1995)'),
 ('cutthroat island', 'Cutthroat Island (1995)'),
 ('casino', 'Casino (1995)'),
 ('sense and sensibility', 'Sense and Sensibility (1995)'),
 ('four rooms', 'Four Rooms (1995)'),
 ('ace ventura: when nature calls', 'Ace Ventura: When Nature Calls (1995)')]

In [16]:
# Find all movies matching the requested_movie		
matches = normalized_title.filter(lambda x: requested_movie.value in x[0])

In [17]:
matches.take(10)

[('father of the bride part ii', 'Father of the Bride Part II (1995)'),
 ('in the name of the father', 'In the Name of the Father (1993)'),
 ('"godfather', '"Godfather'),
 ('father of the bride', 'Father of the Bride (1950)'),
 ('"godfather: part ii', '"Godfather: Part II'),
 ("fathers' day", "Fathers' Day (1997)"),
 ('"godfather: part iii', '"Godfather: Part III'),
 ('this is my father', 'This Is My Father (1998)'),
 ('"grandfather', '"Grandfather'),
 ('"like father', '"Like Father')]

In [18]:
# Collect all the matching titles		
matching_titles = matches.map(lambda x: x[1]).distinct().collect()

In [19]:
# Display the result		
print ('{} Matching titles found:'.format(len(matching_titles)))

134 Matching titles found:


In [20]:
for title in matching_titles:	
    print (title)   

Father of the Bride Part II (1995)
In the Name of the Father (1993)
"Godfather
Father of the Bride (1950)
"Godfather: Part II
Fathers' Day (1997)
"Godfather: Part III
This Is My Father (1998)
"Grandfather
"Like Father
Stepfather II (1989)
"Stepfather
Father Goose (1964)
Beau Pere (a.k.a. Stepfather) (Beau-père) (1981)
How I Killed My Father (a.k.a. My Father and I) (Comment j'ai tué mon Père) (2001)
"Crime of Father Amaro
My Father's Glory (La gloire de mon père) (1990)
"My Father the Hero (Mon père
"Courtship of Eddie's Father
Father of the Bride (1991)
Life with Father (1947)
Father Hood (1993)
Tokyo Godfathers (2003)
3 Godfathers (1948)
My Father the Hero (1994)
Molokai (Molokai: The Story of Father Damien) (1999)
In My Father's Den (2004)
Father's Little Dividend (1951)
Father Sergius (Otets Sergiy) (1917)
Flags of Our Fathers (2006)
My Father and My Son (Babam ve oglum) (2005)
To the Left of the Father (Lavoura Arcaica) (2001)
And When Did You Last See Your Father? (2007)
Dear Zac

In [21]:
sc.stop()